# Just-in-time compilation with numba

Section 14.4 of the notes.

`numba` compiles a Python function to machine code the first time it is called,
for the types of the arguments it was given. Loops that cannot be written as
array operations, such as a time loop, become as fast as compiled code.

In [ ]:
import time
import timeit

import numpy as np
from numba import njit, prange

rng = np.random.default_rng(0)


def best_time(fn, min_total=0.2):
    number, best = 1, float("inf")
    while True:
        t = timeit.timeit(fn, number=number)
        best = min(best, t / number)
        if t >= min_total:
            return best
        number *= 4

## 1. The first call compiles

In [ ]:
@njit
def stencil_numba(u, out):
    d = u.size
    for j in range(d):
        out[j] = u[(j + 1) % d] - 2 * u[j] + u[j - 1]
    return out


d = 1_000_000
u = rng.standard_normal(d)
out = np.empty_like(u)

t0 = time.perf_counter()
stencil_numba(u, out)
t_first = time.perf_counter() - t0
t0 = time.perf_counter()
stencil_numba(u, out)
t_second = time.perf_counter() - t0
print(f"first call  {t_first * 1e3:8.1f} ms   (compiles, then runs)")
print(f"second call {t_second * 1e3:8.1f} ms")
print("compiled for:", stencil_numba.signatures)

A call with arrays of another type compiles a second version.

In [ ]:
stencil_numba(u.astype(np.float32), out.astype(np.float32))
print("compiled for:", stencil_numba.signatures)

## 2. Compared with the array version

The compiled loop makes one pass over the memory and allocates nothing, so it
is faster than the NumPy version, which reads and writes several arrays.

In [ ]:
def stencil_slices(u, out):
    out[1:-1] = u[2:] - 2 * u[1:-1] + u[:-2]
    out[0] = u[1] - 2 * u[0] + u[-1]
    out[-1] = u[0] - 2 * u[-1] + u[-2]
    return out


def stencil_loop(u, out):
    for j in range(u.size):
        out[j] = u[(j + 1) % u.size] - 2 * u[j] + u[j - 1]
    return out


@njit(parallel=True)
def stencil_parallel(u, out):
    d = u.size
    for j in prange(d):
        out[j] = u[(j + 1) % d] - 2 * u[j] + u[j - 1]
    return out


reference = stencil_slices(u, np.empty_like(u))
stencil_parallel(u, out)  # compile before timing
print(
    "all versions agree:",
    np.allclose(reference, stencil_numba(u, np.empty_like(u)))
    and np.allclose(reference, stencil_parallel(u, np.empty_like(u))),
)

for name, fn, number in [
    ("Python loop", stencil_loop, 1),
    ("numpy slices", stencil_slices, 10),
    ("numba", stencil_numba, 10),
    ("numba parallel", stencil_parallel, 10),
]:
    t = min(timeit.repeat(lambda: fn(u, out), repeat=3, number=number)) / number
    print(f"{name:16s} {1e9 * t / d:8.2f} ns per grid point")

## 3. A time loop

The step of Chapter 2 cannot be vectorized over time, since each step needs the
previous one. Compiling the whole loop avoids returning to Python between
steps, and avoids the temporaries of the array version.

In [ ]:
D_COEFF = 2e-5


def step_numpy(u, dt, dx):
    lap = (np.roll(u, -1) - 2 * u + np.roll(u, 1)) / dx**2
    return u + dt * (D_COEFF * lap + u - u**3)


def march_numpy(u, dt, dx, steps):
    for _ in range(steps):
        u = step_numpy(u, dt, dx)
    return u


@njit(cache=True)
def march_numba(u, dt, dx, steps):
    d = u.size
    v = u.copy()
    w = np.empty_like(u)
    for _ in range(steps):
        for j in range(d):
            lap = (v[(j + 1) % d] - 2 * v[j] + v[(j - 1) % d]) / dx**2
            w[j] = v[j] + dt * (D_COEFF * lap + v[j] - v[j] ** 3)
        v, w = w, v
    return v


d = 2048
x = -1 + 2 * np.arange(d) / d
u0 = x**2 * np.cos(np.pi * x)
dx, dt, steps = 2 / d, 0.005, 2_000

a = march_numpy(u0.copy(), dt, dx, steps)
b = march_numba(u0.copy(), dt, dx, steps)  # first call compiles
print("same result:", np.allclose(a, b))

t_np = best_time(lambda: march_numpy(u0.copy(), dt, dx, steps), min_total=2.0)
t_nb = best_time(lambda: march_numba(u0.copy(), dt, dx, steps), min_total=2.0)
print(
    f"numpy {t_np * 1e3:7.1f} ms   numba {t_nb * 1e3:7.1f} ms   ratio {t_np / t_nb:.1f}x"
)

## 4. What numba refuses

Only a part of Python and of NumPy can be compiled. Code that uses objects or
containers of mixed types fails at compilation, with an error that names the
line. It does not fall back to Python silently.

In [ ]:
from numba.core.errors import TypingError


@njit
def uses_a_dict(values):
    seen = {}  # a dictionary of mixed types
    for v in values:
        seen[str(v)] = v
    return len(seen)


try:
    uses_a_dict(np.arange(3.0))
except TypingError as err:
    first_lines = str(err).strip().splitlines()[:3]
    print("compilation failed:")
    print("\n".join(first_lines))